# RENER

In [ ]:
%load_ext autoreload 
%autoreload 2

In [ ]:
import json

# Load jsonl file into a list of dictionaries
with open("../data/labelled/all_parallel.jsonl") as test_f:
    data = [json.loads(line) for line in test_f]

len(data)

In [ ]:
from collections import Counter
from util import extract_ner_labels, flatten
import matplotlib.pyplot as plt

entities = []
for item in data:
    entities.append(extract_ner_labels(item["text"]))

entities = flatten(entities)
entities_dist = Counter([ent[1] for ent in entities])

plt.figure()
plt.pie(
    entities_dist.values(),  # type: ignore
    labels=entities_dist.keys(),  # type: ignore
    autopct=lambda pct: f"{pct:.0f}%" if pct > 1.1 else "",
    startangle=90,
    pctdistance=0.5,
    wedgeprops={"edgecolor": "black"},
)

plt.title("Distribution of NER labels")
plt.show()

In [ ]:
from pathlib import Path
from time import time
from typing import List, Dict, Tuple, Any
import json
import numpy as np
from util import eval_ner, extract_ner_labels
from tqdm.notebook import tqdm
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import Document
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack_integrations.components.generators.ollama import OllamaGenerator
from haystack.document_stores.types import DuplicatePolicy
from prompts import guidelines_llama3_legal_lite, default_examples
from datetime import datetime
from sklearn.model_selection import KFold


class RENERExperiment:
    """"""

    ENTITY_TYPES = ["PER", "LOC", "ORG", "MISC"]

    def __init__(self, model: str, temperature: float, n_splits: int):
        self.model = model
        self.temperature = temperature
        self.n_splits = n_splits
        self.llm = OllamaGenerator(
            model=model,
            url="http://localhost:11434",
            generation_kwargs={"temperature": temperature},
        )
        self.results = {
            "model": model,
            "temperature": temperature,
        }

    def preprocess_entities(
        self,
        entities: List[Tuple[str, str]],
    ) -> List[Tuple[str, str]]:
        """Preprocess entity tuples for consistent comparison."""

        return [(w[0].strip().title(), w[1]) for w in entities]

    def filter_entities_by_type(
        self,
        entities: List[Tuple[str, str]],
        entity_type: str,
    ) -> List[Tuple[str, str]]:
        """Filter entities by type."""

        return [(e[0], e[1]) for e in entities if e[1] == entity_type]

    def build_prompt(self, retrieved_docs: List[Document], query: str) -> str:
        """Build the complete prompt from retrieved documents and query."""
        context = "".join(
            [
                f"Input: {doc.content}\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
                f"Output: {doc.meta['output']}<|eot_id|>\n\n"
                for doc in retrieved_docs
            ]
        ).strip()

        return guidelines_llama3_legal_lite + default_examples + context + query

    def setup_document_store(self, train_data: List[Dict]) -> InMemoryDocumentStore:
        """Initialize and populate document store with training data."""

        document_store = InMemoryDocumentStore()
        docs = [
            Document(content=d["input"], meta={"output": d["text"]}) for d in train_data
        ]
        document_store.write_documents(docs, policy=DuplicatePolicy.OVERWRITE)

        return document_store

    def retrieve_documents(
        self,
        document_store: InMemoryDocumentStore,
        query: str,
        top_k: int,
    ) -> List[Document]:
        """Retrieve top-k documents using BM25."""

        retriever = InMemoryBM25Retriever(document_store=document_store)
        retrieval_result = retriever.run(query=query, top_k=top_k)

        return retrieval_result["documents"]

    def generate_prediction(self, prompt: str) -> str:
        """Generate prediction using LLM."""

        try:
            generation_result = self.llm.run(
                prompt=prompt,
                generation_kwargs={"temperature": self.temperature},
            )
            llm_output = generation_result["replies"][0]
            return llm_output.replace("Output: \n", "")
        except Exception as e:
            print(f"Exception occurred: {e}. Returning empty prediction.")
            return ""

    def evaluate_single_item(
        self,
        item: Dict,
        train_data: List[Dict],
        top_k: int,
    ) -> Dict[str, Any]:
        """Evaluate a single test item."""

        # Define query
        query = f"Input: {item['input']}\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"

        # Setup document store and retrieve
        document_store = self.setup_document_store(train_data)
        retrieved_docs = self.retrieve_documents(document_store, query, top_k)

        # Build prompt and generate
        prompt = self.build_prompt(retrieved_docs, query)
        prediction = self.generate_prediction(prompt)

        if not prediction:
            return None

        item["pred"] = prediction

        # Extract and preprocess entities
        true = self.preprocess_entities(extract_ner_labels(item["text"]))
        pred = self.preprocess_entities(extract_ner_labels(prediction))

        # Overall evaluation
        overall_score = eval_ner(pred, true)

        # Entity-type specific evaluation
        entity_scores = {}
        for entity_type in self.ENTITY_TYPES:
            true_type = self.filter_entities_by_type(true, entity_type)
            pred_type = self.filter_entities_by_type(pred, entity_type)
            entity_scores[entity_type] = eval_ner(pred_type, true_type)

        return {
            "overall": overall_score,
            "by_entity": entity_scores,
            "retrieval_scores": [d.score for d in retrieved_docs],
            "true": true,
            "pred": pred,
        }

    def calculate_fold_metrics(self, scores: List[Dict]) -> Tuple[Dict, Dict]:
        """Calculate average and standard deviation metrics across fold."""
        if not scores:
            return {}, {}

        avg_metrics = {
            key: sum(d[key] for d in scores) / len(scores) for key in scores[0]
        }
        std_metrics = {
            key: float(np.std([d[key] for d in scores])) for key in scores[0]
        }
        return avg_metrics, std_metrics

    def run_fold(
        self, train_data: List[Dict], test_data: List[Dict], top_k: int, fold_idx: int
    ) -> Dict:
        """Run evaluation for a single fold."""
        print(f"Running split # {fold_idx + 1}/{self.n_splits}")

        start_time = time()

        # Store all scores
        all_scores = {
            "overall": [],
            **{entity_type: [] for entity_type in self.ENTITY_TYPES},
            "retrieval": [],
        }

        pbar = tqdm(test_data, desc=f"Fold {fold_idx + 1}")
        for item in pbar:
            result = self.evaluate_single_item(item, train_data, top_k)

            if result is None:
                continue

            all_scores["overall"].append(result["overall"])
            all_scores["retrieval"].append(result["retrieval_scores"])

            for entity_type in self.ENTITY_TYPES:
                all_scores[entity_type].append(result["by_entity"][entity_type])

            # Update progress bar with current metrics
            avg_metrics, _ = self.calculate_fold_metrics(all_scores["overall"])
            pbar.set_postfix(avg_metrics)

        runtime = time() - start_time

        # Calculate metrics for all entity types
        fold_results = {"runtime": runtime, "retrieval_scores": all_scores["retrieval"]}

        for key in ["overall"] + self.ENTITY_TYPES:
            avg, std = self.calculate_fold_metrics(all_scores[key])
            fold_results[f"avg_metrics_{key.lower()}"] = avg
            fold_results[f"std_metrics_{key.lower()}"] = std
            fold_results[f"all_scores_{key.lower()}"] = all_scores[key]

        return fold_results

    def aggregate_topk_metrics(self, top_k: int) -> None:
        """Aggregate metrics across all folds for a given top_k."""
        fold_keys = [f"kfold={i + 1}" for i in range(self.n_splits)]

        for metric_type in ["overall"] + [e.lower() for e in self.ENTITY_TYPES]:
            # Collect average metrics from all folds
            fold_avgs = [
                self.results[f"top_k={top_k}"][fold_key][f"avg_metrics_{metric_type}"]
                for fold_key in fold_keys
            ]

            # Calculate aggregated statistics
            avg_metrics = {
                key: sum(d[key] for d in fold_avgs) / len(fold_avgs)
                for key in fold_avgs[0]
            }
            std_metrics = {
                key: float(np.std([d[key] for d in fold_avgs])) for key in fold_avgs[0]
            }

            suffix = f"_{metric_type}" if metric_type != "overall" else ""
            self.results[f"top_k={top_k}"][f"avg_metrics_topk={top_k}{suffix}"] = (
                avg_metrics
            )
            self.results[f"top_k={top_k}"][f"std_metrics_topk={top_k}{suffix}"] = (
                std_metrics
            )

    def run_experiment(self, data: List[Dict], top_k_range: range) -> None:
        """Run the complete experiment across all top_k values."""
        kf = KFold(n_splits=self.n_splits)

        for top_k in top_k_range:
            print(f"\nRunning experiment with top_k={top_k}")
            self.results[f"top_k={top_k}"] = {}

            for i, (train_idx, test_idx) in enumerate(kf.split(data)):
                train_data = [data[idx] for idx in train_idx]
                test_data = [data[idx] for idx in test_idx]

                fold_results = self.run_fold(train_data, test_data, top_k, i)
                self.results[f"top_k={top_k}"][f"kfold={i + 1}"] = fold_results

            # Aggregate metrics across folds
            self.aggregate_topk_metrics(top_k)

    def save_results(self, output_path: Path) -> None:
        """Save results to JSON file."""

        output_path.parent.mkdir(parents=True, exist_ok=True)

        with open(output_path, "w") as f:
            json.dump(self.results, f, indent=2)

        print(f"\nResults saved to {output_path}")


def main():
    """Main execution function."""
    # Set parameters
    MODEL = "llama3.1:8b"
    TEMPERATURE = 0
    N_SPLITS = 10
    TOP_K_RANGE = range(0, 15)

    # Create output filename
    dt = str(datetime.now()).split(".")[0].replace(" ", "_").replace(":", "-")
    output_filename = Path(f"../output/rener_{dt}.json")

    # Load your data here
    # data = load_data()  # You need to implement this

    # Initialize and run experiment
    experiment = RENERExperiment(
        model=MODEL, temperature=TEMPERATURE, n_splits=N_SPLITS
    )
    experiment.run_experiment(data, TOP_K_RANGE)
    experiment.save_results(output_filename)


if __name__ == "__main__":
    main()

from here on: obsolete, to be fixed

In [ ]:
import seaborn as sns
from util import compute_ner_conf_matrix

conf_matrix = compute_ner_conf_matrix(y_pred, y_true)

labels = ["PER", "LOC", "ORG", "MISC"]
sns.heatmap(
    conf_matrix,
    annot=True,
    cmap="Blues",
    fmt="d",
    xticklabels=labels,
    yticklabels=labels,
)